# Sub-THz Blockage Prediction — LoRA Fine-tuning on Google Colab

Trains the two LoRA architectures (Arch A forecaster, Arch B classifier) on
top of **TimesFM 2.5** and reproduces the paper's metrics + figures, using
Colab's GPU. The `src/` code is identical to `main`; this notebook only
orchestrates it for Colab (GPU-adaptive memory flags, Drive output
persistence, and **resume-safe** experiment loops that survive disconnects).

**Before running:** `Runtime → Change runtime type → GPU` (T4 is fine; A100/
V100 on Colab Pro are faster). Then run the cells top to bottom.

## 1. Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU — set Runtime → Change runtime type → GPU'
p = torch.cuda.get_device_properties(0)
print(f'{p.name}  |  {p.total_memory/1e9:.1f} GB  |  torch {torch.__version__}')
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## 2. Clone the repo (code + data) and install deps

The dataset is committed in the repo, so a shallow clone brings everything.
For a **private** repo, paste a GitHub token (Settings → Developer settings →
Personal access tokens); leave it blank if the repo is public.

In [ ]:
REPO = 'github.com/kaefcatcher/THz_blockage.git'
BRANCH = 'collab'
TOKEN = ''  # e.g. 'ghp_...'; leave '' for a public repo

import os
url = f"https://{TOKEN + '@' if TOKEN else ''}{REPO}"
if not os.path.isdir('THz_blockage'):
    !git clone --depth 1 --branch {BRANCH} {url}
%cd THz_blockage
# torch is preinstalled on Colab; TimesFM 2.5 needs transformers >= 5.12
!pip install -q 'transformers>=5.12' 'peft>=0.13' 'safetensors>=0.4' einops
import transformers, peft; print('transformers', transformers.__version__, '| peft', peft.__version__)

## 3. (Optional) Mount Google Drive so checkpoints/results survive disconnects

Recommended for the long experiments. Skips cleanly if you decline. Outputs
are symlinked into a Drive folder, so re-running a later session picks up
exactly where it left off.

In [ ]:
USE_DRIVE = True  # set False to keep outputs only in the ephemeral Colab VM
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OUT = '/content/drive/MyDrive/thz_blockage_outputs'
    os.makedirs(DRIVE_OUT, exist_ok=True)
    if os.path.islink('outputs') or not os.path.exists('outputs'):
        if os.path.islink('outputs'): os.unlink('outputs')
        os.symlink(DRIVE_OUT, 'outputs')
    else:
        import shutil
        for sub in ('checkpoints', 'results', 'figures'):
            os.makedirs(f'{DRIVE_OUT}/{sub}', exist_ok=True)
    print('outputs ->', os.path.realpath('outputs'))
else:
    os.makedirs('outputs/results', exist_ok=True)
    print('outputs in ephemeral VM storage')

## 4. Verify the data pipeline and LoRA setup

These are the spec's Step-1 and Step-2 checkpoints. The first model load
downloads the TimesFM 2.5 weights (~0.9 GB) into the Colab cache.

In [ ]:
!python src/data.py
!python src/lora_common.py

## 5. GPU-adaptive memory flags

Smaller GPUs (T4, 16 GB) use bf16 + per-layer gradient checkpointing. Big GPUs
(A100/V100, 40 GB) can run a larger batch. These don't change results —
accumulation preserves the effective batch; bf16 is a minor numerical change.

In [ ]:
import torch
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if gb < 24:  # T4 / V100-16
    CLI = '--bf16 --grad-checkpoint --batch-size 16 --accum-steps 4'
    TRAIN_KWARGS = dict(dtype='bf16', grad_checkpoint=True, batch_size=16, accum_steps=4)
else:        # A100 / V100-32+
    CLI = '--bf16 --batch-size 64'
    TRAIN_KWARGS = dict(dtype='bf16', batch_size=64)
print('GPU', f'{gb:.0f}GB', '->', CLI)

## 6. Train the canonical Arch A + Arch B checkpoints (N=216)

Saved to `outputs/checkpoints/` (on Drive if mounted). Watch the printed
`peak=X.XXGB` per epoch — if it OOMs, drop to `--low-mem` or a smaller
`--batch-size`. Each is a full 30-epoch run with early stopping.

In [ ]:
!python src/lora_arch_a.py {CLI}

In [ ]:
!python src/lora_arch_b.py --loss bce {CLI}
!python src/lora_arch_b.py --loss focal {CLI}

## 7. Main metrics table (Step 5)

Zero-shot row is read from the committed parquet; the four Arch rows are
trained here. Writes `outputs/results/metrics_main.csv`.

In [ ]:
import sys; sys.path.insert(0, 'src')
import experiments
experiments.build_metrics_main(train_kwargs=TRAIN_KWARGS)

## 8. Data-efficiency experiment — **resume-safe** (Step 6)

42 runs (7 sizes x 3 seeds x 2 archs). This is the long pole — likely longer
than one free-Colab session. The loop **skips runs already in the CSV**, so
just re-run this cell in a new session to continue. For a fast first look,
set `SEEDS = [0]` and/or trim `N_GRID`.

In [ ]:
import os, pandas as pd, sys
sys.path.insert(0, 'src')
import data, importlib, experiments; importlib.reload(experiments)
from experiments import run_one, N_GRID, SEEDS, TW

EFF = 'outputs/results/metrics_data_efficiency.csv'
os.makedirs('outputs/results', exist_ok=True)

def _done(csv, cols):
    if not os.path.exists(csv): return set()
    d = pd.read_csv(csv)
    return {tuple(r) for r in d[cols].itertuples(index=False, name=None)}

pool = data.get_train_pool_files()
done = _done(EFF, ['arch', 'N_traces', 'seed'])
for N in N_GRID:
    for seed in SEEDS:
        files = data.sample_traces_stratified(pool, N, seed=seed)
        for arch in ('A', 'B'):
            if (arch, N, seed) in done:
                print(f'skip arch={arch} N={N} seed={seed} (already done)'); continue
            m = run_one(arch, files, seed=seed, loss='bce', train_kwargs=TRAIN_KWARGS)
            row = {'arch': arch, 'N_traces': N, 'seed': seed, 'Tw_ms': TW,
                   'acc': round(m['acc'],4), 'precision': round(m['precision'],4),
                   'recall': round(m['recall'],4), 'f1': round(m['f1'],4)}
            pd.DataFrame([row]).to_csv(EFF, mode='a', header=not os.path.exists(EFF), index=False)
            print(f"done arch={arch} N={N} seed={seed} f1={m['f1']:.4f}")
print('data-efficiency complete' )

## 9. Diversity experiment — **resume-safe** (Step 7)

Balanced (10/config x 8) vs homogeneous (configs 1-2 only), Arch B / BCE, 3
seeds. Same resume behavior: re-run to continue. Writes
`outputs/results/metrics_diversity.csv`.

In [ ]:
import os, pandas as pd, sys, tempfile
sys.path.insert(0, 'src')
import data, evaluate as ev, lora_arch_b
from experiments import _sample_config, _div_row, report_diversity, TW

DIV = 'outputs/results/metrics_diversity.csv'
pool = data.get_train_pool_files(); eval_files = data.get_eval_files()
theta = data.get_theta(); cfgs = sorted({data.meta_of(p)['set'] for p in eval_files})
done = set()
if os.path.exists(DIV):
    _d = pd.read_csv(DIV)
    done = {tuple(r) for r in _d[['condition', 'seed']].itertuples(index=False, name=None)}

for seed in [0, 1, 2]:
    conds = {'balanced': data.sample_traces_stratified(pool, 80, seed=seed),
             'homogeneous': _sample_config(pool, 1, 40, seed) + _sample_config(pool, 2, 40, seed)}
    for cond, files in conds.items():
        if (cond, seed) in done:
            print(f'skip {cond} seed={seed} (done)'); continue
        with tempfile.TemporaryDirectory() as tmp:
            model, _f1, _t = lora_arch_b.train(loss_kind='bce', train_files=files, seed=seed,
                                               save_dir=__import__('pathlib').Path(tmp),
                                               enforce_gate=False, **TRAIN_KWARGS)
            rows = [_div_row(cond, seed, 'overall', ev.evaluate(model, eval_files, TW, theta, 'arch_b'))]
            for c in cfgs:
                ef = data.files_for_configs(eval_files, [c])
                rows.append(_div_row(cond, seed, str(c), ev.evaluate(model, ef, TW, theta, 'arch_b')))
        pd.DataFrame(rows).to_csv(DIV, mode='a', header=not os.path.exists(DIV), index=False)
        print(f'done {cond} seed={seed}')
        del model
report_diversity(pd.read_csv(DIV))

## 10. Figures (Step 8)

`--with-models` loads the trained Arch A/B checkpoints for the real PR curve.
Produces the three paper PDFs under `outputs/figures/`.

In [ ]:
!python src/figures.py --with-models
from IPython.display import IFrame, display
for f in ('data_efficiency_curve', 'pr_curve', 'metrics_table'):
    print(f'outputs/figures/{f}.pdf')

## 11. Save / export results

If you mounted Drive, everything under `outputs/` is already persisted. To pull
the small artifacts (checkpoints + CSVs + figures, a few MB) to your machine,
zip and download — or commit them back to the `collab` branch.

In [ ]:
import shutil
shutil.make_archive('/content/thz_outputs', 'zip', 'outputs')
from google.colab import files
files.download('/content/thz_outputs.zip')

# --- or push results back to the collab branch (needs a token with repo scope) ---
# !git config user.email 'you@example.com' && git config user.name 'you'
# !git add outputs && git commit -m 'colab: trained checkpoints + metrics + figures'
# !git push https://{TOKEN}@{REPO} HEAD:collab